# Data Exploration Part 3: Charts and Visualizations
## Kairos Project - Fadel Transportes Fleet Maintenance Prediction

**Team:** Ilariê  
**Objective:** Generate minimum 3 charts showing relationships between key variables  
**Focus:** Vehicle brand total costs, branch × maintenance frequency, product total costs  
**Tools:** matplotlib, seaborn with proper legends and titles

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully!")

In [ ]:
# Load all datasets
try:
    df_service = pd.read_excel('data/SERVICE_ORDER_BASE.xlsx')
    df_service = pd.read_excel('data/SERVICE_ORDER_BASE.xlsx')
    df_vehicles = pd.read_excel('data/VEHICLES_BASE.xlsx')

    print("=== DATASETS LOADED ===")
    print(f"Main Dataset: {df_service.shape[0]:,} records × {df_service.shape[1]} columns")
    print(f"Service Orders: {df_service.shape[0]:,} records × {df_service.shape[1]} columns")
    print(f"Vehicle Master: {df_vehicles.shape[0]:,} records × {df_vehicles.shape[1]} columns")

    print("\nMain dataset columns:")
    for i, col in enumerate(df_service.columns, 1):
        print(f"{i:2}. {col}")

except Exception as e:
    print(f"Error loading datasets: {e}")
    print("Please ensure data files are in the correct location.")

## 2. Chart 1: Vehicle Brands with Highest Total Maintenance Costs

In [ ]:
# Analyze total maintenance costs by vehicle brand from service orders
# Check what columns are available for brand information
print("Checking brand-related columns in service dataset:")
brand_cols = [col for col in df_service.columns if 'MANUFACT' in col.upper() or 'BRAND' in col.upper() or 'MAKE' in col.upper()]
print("Brand columns found:", brand_cols)

# Use manufacturer name column for brand analysis
brand_column = 'MANUFACTURER NAME'
cost_column = 'GRAND TOTAL'

if brand_column in df_service.columns and cost_column in df_service.columns:
    # Filter valid data
    df_brand_analysis = df_service[
        (df_service[cost_column] > 0) &
        (df_service[cost_column] < 100000) &  # Remove extreme outliers
        (df_service[brand_column].notna())
    ].copy()

    # Calculate brand statistics - focus on TOTAL costs
    brand_stats = df_brand_analysis.groupby(brand_column)[cost_column].agg([
        'sum', 'count', 'mean', 'median'
    ]).round(2)

    # Filter brands with at least 20 maintenance events for statistical significance
    significant_brands = brand_stats[brand_stats['count'] >= 20]
    # Sort by TOTAL cost (sum) instead of average
    top_brands = significant_brands.sort_values('sum', ascending=False).head(15)

    # Create horizontal bar chart
    plt.figure(figsize=(14, 12))
    colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(top_brands)))
    bars = plt.barh(range(len(top_brands)), top_brands['sum'], color=colors,
                    edgecolor='black', linewidth=0.8)

    plt.title('Top Vehicle Brands by Total Maintenance Cost\n(Brands with more than 20 maintenance events)',
             fontsize=16, weight='bold', pad=20)
    plt.xlabel('Total Maintenance Cost (R$)', fontsize=14)
    plt.ylabel('Vehicle Brand (Manufacturer)', fontsize=14)

    # Customize y-axis
    plt.yticks(range(len(top_brands)), top_brands.index)

    # Add value labels with count information
    for i, (bar, total_cost, count, avg_cost) in enumerate(zip(bars, top_brands['sum'], top_brands['count'], top_brands['mean'])):
        # Show total cost
        label_text = f'R$ {total_cost:,.0f}'
        plt.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2 + 0.1,
                 label_text, ha='left', va='center', fontsize=10, weight='bold')

        # Add event count and average
        count_text = f'({count:,} events, avg R$ {avg_cost:,.0f})'
        plt.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2 - 0.1,
                 count_text, ha='left', va='center', fontsize=9)

    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

    # Additional analysis - create a summary table
    print("\nDETAILED BRAND ANALYSIS (by Total Cost):")
    print(f"{'Brand':<25} {'Total Cost':<15} {'Events':<8} {'Avg Cost':<12} {'Median':<10}")
    print("-" * 75)
    for brand, data in top_brands.iterrows():
        print(f"{brand[:23]:<25} R$ {data['sum']:>11,.0f}   {data['count']:>6.0f}   R$ {data['mean']:>8.2f}   R$ {data['median']:>6.2f}")

    print(f"\nCHART 1 INSIGHTS:")
    highest_total = top_brands.index[0]
    highest_total_cost = top_brands['sum'].iloc[0]
    lowest_total = top_brands.index[-1]
    lowest_total_cost = top_brands['sum'].iloc[-1]

    print(f" Highest total cost brand: {highest_total} (R$ {highest_total_cost:,.0f})")
    print(f" Lowest total cost (top 15): {lowest_total} (R$ {lowest_total_cost:,.0f})")
    print(f" Cost difference: R$ {highest_total_cost - lowest_total_cost:,.0f}")
    print(f" Total brands analyzed: {len(significant_brands)} (with ≥20 events)")
    print(f" Combined total cost (top 15): R$ {top_brands['sum'].sum():,.0f}")

    # Business insights
    total_fleet_cost = top_brands['sum'].sum()
    top_5_cost = top_brands['sum'].head(5).sum()
    top_5_percentage = (top_5_cost / total_fleet_cost) * 100

    print(f" Top 5 brands represent {top_5_percentage:.1f}% of total maintenance costs")
    print(f" This concentration suggests focusing resources on high-cost brands")

else:
    print(f"Required columns not found. Available service dataset columns:")
    print(list(df_service.columns)[:10], "...")

## 3. Chart 2: Branch vs Maintenance Frequency Analysis

In [ ]:
# Analyze maintenance frequency by branch
branch_counts = df_service['FILIAL DO SISTEMA'].value_counts().head(10)

plt.figure(figsize=(14, 8))
colors = sns.color_palette('viridis', len(branch_counts))
bars = plt.bar(range(len(branch_counts)), branch_counts.values, color=colors,
               edgecolor='black', linewidth=0.7)

plt.title('Top Branches by Maintenance Frequency', fontsize=16, weight='bold', pad=20)
plt.xlabel('Branch (FILIAL DO SISTEMA)', fontsize=14)
plt.ylabel('Number of Maintenance Events', fontsize=14)

# Customize x-axis
plt.xticks(range(len(branch_counts)), branch_counts.index, rotation=45, ha='right')

# Add value labels on bars
for i, (bar, value) in enumerate(zip(bars, branch_counts.values)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{value:,}', ha='center', va='bottom', fontweight='bold')

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Analysis
total_events = len(df_service)
top3_events = branch_counts.head(3).sum()
top3_percentage = (top3_events / total_events) * 100

print(f"CHART 2 INSIGHTS:")
print(f" Top branch: {branch_counts.index[0]} with {branch_counts.iloc[0]:,} events")
print(f" Total branches analyzed: {df_service['FILIAL DO SISTEMA'].nunique():,}")
print(f" Top 3 branches handle {top3_events:,} events ({top3_percentage:.1f}% of total)")
print(f" This concentration suggests operational differences between branches")
print(f" Branch {branch_counts.index[0]} may need resource optimization or investigation")

## 4. Chart 3: Products with Highest Total Maintenance Costs

In [ ]:
# Analyze total costs by product type (using main dataset)
cost_by_product = df_service.groupby('DESCRICAO PRODUTO')['CUSTO TOTAL'].agg([
    'sum', 'count', 'mean', 'median'
]).round(2)

# Filter products with at least 10 occurrences for statistical significance
significant_products = cost_by_product[cost_by_product['count'] >= 10]
# Sort by TOTAL cost (sum) instead of average
top_products = significant_products.sort_values('sum', ascending=False).head(15)

plt.figure(figsize=(14, 12))
colors = sns.color_palette('viridis', len(top_products))
bars = plt.barh(range(len(top_products)), top_products['sum'], color=colors,
                edgecolor='black', linewidth=0.7)

plt.title('Top 15 Products by Total Maintenance Cost\n(Products with more than 10 occurrences)',
         fontsize=16, weight='bold', pad=20)
plt.xlabel('Total Maintenance Cost (R$)', fontsize=14)
plt.ylabel('Product Description', fontsize=14)

# Customize y-axis
plt.yticks(range(len(top_products)), top_products.index)

# Add value labels with count and average information
for i, (bar, total_cost, count, avg_cost) in enumerate(zip(bars, top_products['sum'], top_products['count'], top_products['mean'])):
    # Show total cost
    total_text = f'R$ {total_cost:,.0f}'
    plt.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2 + 0.1,
             total_text, ha='left', va='center', fontsize=10, weight='bold')

    # Add count and average info
    details_text = f'({count:,} events, avg R$ {avg_cost:.0f})'
    plt.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2 - 0.1,
             details_text, ha='left', va='center', fontsize=9)

plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# Additional analysis
print("\nDETAILED PRODUCT ANALYSIS (by Total Cost):")
print(f"{'Product':<30} {'Total Cost':<15} {'Events':<8} {'Avg Cost':<12} {'Median':<10}")
print("-" * 80)
for product, data in top_products.iterrows():
    print(f"{product[:28]:<30} R$ {data['sum']:>11,.0f}   {data['count']:>6.0f}   R$ {data['mean']:>8.2f}   R$ {data['median']:>6.2f}")

print(f"\nCHART 3 INSIGHTS:")
highest_product = top_products.index[0]
highest_product_cost = top_products['sum'].iloc[0]
lowest_product = top_products.index[-1]
lowest_product_cost = top_products['sum'].iloc[-1]

print(f" Highest total cost product: {highest_product}")
print(f" Total cost: R$ {highest_product_cost:,.0f}")
print(f" Lowest total cost (top 15): {lowest_product} (R$ {lowest_product_cost:,.0f})")
print(f" Cost difference: R$ {highest_product_cost - lowest_product_cost:,.0f}")
print(f" Total unique products: {df_service['DESCRICAO PRODUTO'].nunique():,}")
print(f" Combined cost (top 15): R$ {top_products['sum'].sum():,.0f}")

# Business concentration analysis
total_product_cost = top_products['sum'].sum()
top_5_product_cost = top_products['sum'].head(5).sum()
top_5_product_percentage = (top_5_product_cost / total_product_cost) * 100

print(f" Top 5 products represent {top_5_product_percentage:.1f}% of top 15 total costs")
print(f" Focus procurement negotiations on high-total-cost items")